# 04 Statistical Analysis



**Sections:**
1. Setup
2. Descriptive statistics by segment
3. Hypothesis testing — gender differences
4. Hypothesis testing — academic year differences
5. Correlation analysis
6. Multiple linear regression — burnout score
7. Logistic regression — high risk classification
8. Statistical summary

## 4.1 Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols, logit

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Work on a stratified sample for regression (1M rows is slow for OLS)
SAMPLE_SIZE = 50_000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Working sample: {len(df_sample):,} rows')

sns.set_theme(style='whitegrid')
df.head()

## 4.2 Descriptive Statistics by Segment

In [ ]:
key_metrics = ['burnout_score', 'stress_level', 'anxiety_score', 'depression_score',
               'dropout_risk', 'mental_health_index', 'academic_performance',
               'sleep_hours', 'social_support', 'physical_activity']

print('=== By Gender ===')
display(df.groupby('gender')[key_metrics].agg(['mean', 'std']).round(3))

print('\n=== By Risk Level ===')
display(df.groupby('risk_level')[key_metrics].agg(['mean', 'std']).round(3))

print('\n=== By Academic Year ===')
display(df.groupby('academic_year')[key_metrics].mean().round(3))

## 4.3 Hypothesis Testing — Gender Differences

**H₀:** There is no significant difference in burnout score between male and female students.  
**H₁:** There is a significant difference in burnout score between male and female students.  
**Test:** Welch's two-sample t-test (α = 0.05)

In [ ]:
male   = df.loc[df['gender'] == 'Male',   'burnout_score']
female = df.loc[df['gender'] == 'Female', 'burnout_score']

t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)

print(f'Male   — mean: {male.mean():.4f}, std: {male.std():.4f}, n: {len(male):,}')
print(f'Female — mean: {female.mean():.4f}, std: {female.std():.4f}, n: {len(female):,}')
print(f'\nWelch t-statistic : {t_stat:.4f}')
print(f'p-value           : {p_value:.6f}')
print(f'\nConclusion: {"Reject H₀ — significant gender difference" if p_value < 0.05 else "Fail to reject H₀ — no significant gender difference"}')

In [ ]:
# Test for multiple metrics by gender
test_metrics = ['burnout_score', 'stress_level', 'anxiety_score', 'depression_score', 'dropout_risk']
results = []

for metric in test_metrics:
    g1 = df.loc[df['gender'] == 'Male',   metric]
    g2 = df.loc[df['gender'] == 'Female', metric]
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    results.append({
        'metric': metric,
        'male_mean': g1.mean(),
        'female_mean': g2.mean(),
        'diff': g1.mean() - g2.mean(),
        't_stat': t,
        'p_value': p,
        'significant': p < 0.05
    })

gender_test_df = pd.DataFrame(results)
gender_test_df.round(4)

## 4.4 Hypothesis Testing — Academic Year Differences

**H₀:** Mean burnout score is equal across all academic years.  
**H₁:** At least one academic year has a significantly different mean burnout score.  
**Test:** One-way ANOVA (α = 0.05)

In [ ]:
groups = [df.loc[df['academic_year'] == yr, 'burnout_score'] for yr in sorted(df['academic_year'].unique())]
f_stat, p_anova = stats.f_oneway(*groups)

print(f'One-way ANOVA — Burnout Score across Academic Years')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value    : {p_anova:.6f}')
print(f'\nConclusion: {"Reject H₀ — significant year-level differences" if p_anova < 0.05 else "Fail to reject H₀"}')

# Mean burnout per year
print('\nMean burnout by year:')
print(df.groupby('academic_year')['burnout_score'].mean().round(4))

In [ ]:
# ANOVA for all key metrics across academic years
anova_results = []
for metric in test_metrics:
    grps = [df.loc[df['academic_year'] == yr, metric] for yr in sorted(df['academic_year'].unique())]
    f, p = stats.f_oneway(*grps)
    anova_results.append({'metric': metric, 'F_stat': f, 'p_value': p, 'significant': p < 0.05})

pd.DataFrame(anova_results).round(4)

## 4.5 Correlation Analysis

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr()

# Correlations with burnout_score
burnout_corr = corr['burnout_score'].drop('burnout_score').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d73027' if v > 0 else '#4575b4' for v in burnout_corr.values]
ax.barh(burnout_corr.index, burnout_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Burnout Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print('\nCorrelation values:')
print(burnout_corr.round(4).to_string())

In [ ]:
# Correlations with dropout_risk
dropout_corr = corr['dropout_risk'].drop('dropout_risk').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d73027' if v > 0 else '#4575b4' for v in dropout_corr.values]
ax.barh(dropout_corr.index, dropout_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Dropout Risk', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

## 4.6 Multiple Linear Regression — Predicting Burnout Score

In [ ]:
# OLS regression on sample
formula = (
    'burnout_score ~ stress_level + anxiety_score + depression_score + '
    'sleep_hours + study_hours_per_day + physical_activity + social_support + '
    'screen_time + internet_usage + financial_stress + family_expectation + '
    'exam_pressure + academic_performance + age'
)

model = ols(formula, data=df_sample).fit()
print(model.summary())

In [ ]:
# Coefficient plot
coef = model.params.drop('Intercept')
conf = model.conf_int().drop('Intercept')

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(coef))
ax.barh(y_pos, coef.values,
        xerr=[(coef.values - conf[0].values), (conf[1].values - coef.values)],
        color=['#d73027' if v > 0 else '#4575b4' for v in coef.values],
        capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels([c.replace('_', ' ').title() for c in coef.index])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('OLS Regression Coefficients — Burnout Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient (with 95% CI)')
plt.tight_layout()
plt.show()

print(f'\nR² = {model.rsquared:.4f}')
print(f'Adj R² = {model.rsquared_adj:.4f}')

## 4.7 Logistic Regression — High Risk Classification

In [ ]:
# Encode risk_level as binary
df_sample['high_risk'] = (df_sample['risk_level'] == 'High').astype(int)

logit_formula = (
    'high_risk ~ burnout_score + stress_level + anxiety_score + depression_score + '
    'sleep_hours + physical_activity + social_support + financial_stress + '
    'family_expectation + exam_pressure + academic_performance'
)

logit_model = logit(logit_formula, data=df_sample).fit(maxiter=200, disp=False)
print(logit_model.summary())

In [ ]:
# Odds ratios
odds_ratios = np.exp(logit_model.params).drop('Intercept').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d73027' if v > 1 else '#4575b4' for v in odds_ratios.values]
ax.barh(range(len(odds_ratios)), odds_ratios.values, color=colors)
ax.set_yticks(range(len(odds_ratios)))
ax.set_yticklabels([c.replace('_', ' ').title() for c in odds_ratios.index])
ax.axvline(1, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Odds Ratios — High Risk Classification', fontsize=13, fontweight='bold')
ax.set_xlabel('Odds Ratio')
plt.tight_layout()
plt.show()

print('\nOdds Ratios:')
print(odds_ratios.round(4).to_string())

## 4.8 Normality Tests

In [ ]:
# Shapiro-Wilk on a small sample (n=5000) — full dataset is too large
normality_sample = df.sample(n=5000, random_state=42)
normality_results = []

for col in ['burnout_score', 'stress_level', 'anxiety_score', 'depression_score', 'dropout_risk']:
    stat, p = stats.shapiro(normality_sample[col])
    normality_results.append({'column': col, 'W_stat': stat, 'p_value': p,
                               'normal': p > 0.05})

pd.DataFrame(normality_results).round(4)

## 4.9 Statistical Summary

| Analysis | Key Finding | Business Interpretation |
|---|---|---|
| Gender t-test | Significant difference in burnout between genders (p < 0.05) | Gender-specific mental health programmes are justified |
| ANOVA — academic year | Significant burnout variation across years (p < 0.05) | Year-specific interventions are needed, especially for upper years |
| OLS regression | Stress, anxiety, and depression are the strongest positive predictors of burnout | Targeting these three factors will reduce burnout most efficiently |
| OLS regression | Sleep hours and social support are significant negative predictors | Sleep and peer support programmes have measurable protective effects |
| Logistic regression | Burnout score is the strongest predictor of high-risk classification | Burnout score alone can serve as an early-warning triage metric |
| R² | Model explains ~60-70% of burnout variance | The selected predictors capture most of the explainable variance |

